# Test Notebook for push_to_kindle.py

This notebook is for testing the functions in `push_to_kindle.py`. Make sure you have installed the required packages:

```bash
pip install rarfile Pillow requests
```

Before running the tests, make sure to replace the placeholder `test.cbr` file with a real CBR file.

In [11]:
import os
import sys
import tempfile
import zipfile
from PIL import Image
import importlib

# Add the current directory to the path to import the script
sys.path.append('.')

import push_to_kindle

__file__='.'

## 1. Test `convert_cbr_to_cbz`

In [12]:
importlib.reload(push_to_kindle)
temp_dir_cbr = os.path.join(os.path.dirname(__file__), 'tmp')
os.makedirs(temp_dir_cbr, exist_ok=True)
test_cbr_path = './test.cbr' # Make sure this file exists

# Before running this, create a dummy rar file named test.cbr
# You can use a command like `rar a test.cbr test_file.txt`
if os.path.exists(test_cbr_path):
    converted_cbz_path = push_to_kindle.convert_cbr_to_cbz(test_cbr_path, temp_dir_cbr)
    print(f"File converted to: {converted_cbz_path}")
    # You can inspect the temp_dir_cbr to see the result
else:
    print(f"Test file not found: {test_cbr_path}")

Converting CBR to CBZ: ./test.cbr
Converted to: tmp/test.cbz
File converted to: tmp/test.cbz


## 2. Test `convert_images_to_jpg`

In [13]:
importlib.reload(push_to_kindle)
temp_dir_img = os.path.join(os.path.dirname(__file__), 'tmp')
os.makedirs(temp_dir_img, exist_ok=True)

# Create a dummy CBZ with a WEBP image for testing
dummy_cbz_path = os.path.join(temp_dir_img, 'test.cbz')
dummy_webp_path = os.path.join(temp_dir_img, 'test.webp')

# Create a dummy webp image
img = Image.new('RGB', (100, 100), color = 'red')
img.save(dummy_webp_path, 'WEBP')

with zipfile.ZipFile(dummy_cbz_path, 'w') as zf:
    zf.write(dummy_webp_path, arcname='page1.webp')

converted_jpg_cbz_path = push_to_kindle.convert_images_to_jpg(dummy_cbz_path, temp_dir_img)
print(f"CBZ with converted images: {converted_jpg_cbz_path}")

# Verify the content of the new cbz
with zipfile.ZipFile(converted_jpg_cbz_path, 'r') as zf:
    for name in zf.namelist():
        print(f"  - {name}")

Checking images in: tmp/test.cbz
Image conversion needed. Converting to JPG...
Converted images and saved to: tmp/test_converted.cbz
CBZ with converted images: tmp/test_converted.cbz
  - page1.jpg


## 3. Test `push_to_kindle`

This test will attempt to upload a file to your File Browser instance. 
**Make sure your File Browser is running and the credentials in `push_to_kindle.py` are correct.**

In [20]:
importlib.reload(push_to_kindle)
# Create a dummy file to upload
temp_dir_push = os.path.join(os.path.dirname(__file__), 'tmp')
os.makedirs(temp_dir_push, exist_ok=True)
file_to_push = os.path.join(temp_dir_push, 'test_book.cbz')

with zipfile.ZipFile(file_to_push, 'w') as zf:
    zf.writestr('info.txt', 'This is a test file.')

print(f"Attempting to push: {file_to_push}")
push_to_kindle.push_to_kindle(file_to_push)
print("Test finished.")

Attempting to push: tmp/test_book.cbz
--- Debug: Pushing to Kindle ---
File path: tmp/test_book.cbz
Attempting to login to: http://192.168.29.55/api/login
Login response status code: 403
Error logging into File Browser: 403 Client Error: Forbidden for url: http://192.168.29.55/api/login
Test finished.
